In [0]:
# %pip uninstall -y cudf-cu13 cuml-cu13 cugraph-cu13 cuvs-cu13 libcuml-cu13 libcuvs-cu13 libraft-cu13 pylibcudf-cu13 pylibraft-cu13 rmm-cu13

# Install PyTorch with CUDA 12.9
%pip install --upgrade --no-cache-dir \
  --index-url https://download.pytorch.org/whl/cu129 \
  "torch==2.9.1" "torchvision==0.24.1" "torchaudio==2.9.1"

# Install CUDA-12 RAPIDS packages, pinned to the same RAPIDS release
%pip install --upgrade --no-cache-dir \
  "cupy-cuda12x>=13.6.0" \
  "rmm-cu12==26.02.*" \
  "cudf-cu12==26.02.*" \
  "cuml-cu12[dask]==26.02.*" \
  "dask==2026.1.1" \
  "distributed==2026.1.1" \
  "dask-cuda==26.02.*" \
  "dask-cudf-cu12==26.02.*"

%pip install faiss-gpu-cu12 optuna sqlalchemy

dbutils.library.restartPython()

In [0]:
import cuml
%reload_ext cuml.accel

In [0]:

import cuml
import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
pio.renderers.default = "browser"


from multiprocessing import Manager
from cuml.dask.manifold import UMAP
from cuml.dask.cluster import KMeans  # GPU-accelerated KMeans
from pathlib import Path
import dask_cudf

import glob
import time
from tqdm import tqdm
import psutil

import cupy as cp
import pandas as pd


In [0]:
# ----------------------------

# CONFIG (same as your tag logic)

# ----------------------------

cache_dir = Path("/dbfs/tmp/pftsleep_cache")

encoder_name = "PFTSleep"

num_files = 1229

frequency = 125

win_length = 750

hop_length = 750

max_seq_len_sec = 8 * 3600

def cache_tag(encoder_name, num_files, frequency, win_length, hop_length, max_seq_len_sec):

    return f"{encoder_name}__files{num_files}__freq{frequency}__win{win_length}__hop{hop_length}__max{max_seq_len_sec}"

tag = cache_tag(encoder_name, num_files, frequency, win_length, hop_length, max_seq_len_sec)

shard_dir = cache_dir / f"{tag}_shards"

shard_files = sorted(glob.glob(str(shard_dir / "Z_part_*.npy")))

assert len(shard_files) > 0, f"No shards found in {shard_dir}"

print(f"Found {len(shard_files)} shards")

print(f"RAM now: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# ----------------------------

# 1) Compute total rows cheaply (mmap_mode avoids loading)

# ----------------------------

D = 512

total_rows = 0

first = np.load(shard_files[0], mmap_mode="r")

print("Example shard shape/dtype:", first.shape, first.dtype)

for f in shard_files:

    total_rows += np.load(f, mmap_mode="r").shape[0]

print(f"Total rows: {total_rows:,}  (expected ~5,899,200)")

print(f"Approx raw size float16: {total_rows * D * 2 / 1e9:.2f} GB")

print(f"Approx raw size float32: {total_rows * D * 4 / 1e9:.2f} GB")

# ----------------------------

# 2) Build a memmap on local NVMe (fast + avoids RAM ceilings)

# ----------------------------

local_dir = Path("/local_disk0/pftsleep_memmap")

local_dir.mkdir(parents=True, exist_ok=True)

mm_path = local_dir / f"X__{tag}__l2norm_f32.memmap"

shape_path = local_dir / f"X__{tag}__shape.txt"

# Create/overwrite memmap file

X_mm = np.memmap(mm_path, dtype=np.float32, mode="w+", shape=(total_rows, D))

# ----------------------------

# 3) Fill memmap sequentially (no vstack, no giant allocations)

# ----------------------------

t0 = time.time()

offset = 0

for f in tqdm(shard_files, desc="Writing memmap", unit="file"):

    shard = np.load(f)  # should be float16

    if shard.dtype != np.float16:

        # still fine; we cast below, but this warns you if storage isn't what you expect

        pass

    n = shard.shape[0]

    X_mm[offset:offset+n, :] = shard.astype(np.float32, copy=False)

    offset += n

X_mm.flush()

t1 = time.time()

with open(shape_path, "w") as s:

    s.write(f"{total_rows},{D}\n")

print(f"\nMemmap written: {mm_path}")

print(f"Write time: {t1 - t0:.2f} sec")

print(f"RAM now: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# ----------------------------

# 4) In-place L2 normalize in chunks (still memmap-backed)

# ----------------------------

print("\nNormalizing memmap in chunks...")

chunk_rows = 250_000  # tune if you want (100k–500k is fine)

eps = 1e-8

t2 = time.time()

for start in tqdm(range(0, total_rows, chunk_rows), desc="L2 normalize", unit="chunk"):

    end = min(total_rows, start + chunk_rows)

    block = X_mm[start:end, :]  # view into memmap (does not load everything)

    norms = np.linalg.norm(block, axis=1, keepdims=True)

    block /= np.maximum(norms, eps)

X_mm.flush()

t3 = time.time()

print(f"Normalization time: {t3 - t2:.2f} sec")

print("✅ Memmap X is ready. Use X_mm like a normal array: X_mm[i:j]")

print(f"RAM now: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# ----------------------------

# 5) Load your metadata normally (small)

# ----------------------------

night_id = np.load(cache_dir / f"night_id__{tag}.npy")

time_idx = np.load(cache_dir / f"time_idx__{tag}.npy")

zarr_file_idx = np.load(cache_dir / f"zarr_file_idx__{tag}.npy")

print("Metadata loaded:", night_id.shape, time_idx.shape, zarr_file_idx.shape)
 

In [0]:
# -------------------------
# Restructure demographics CSV to match zarr folder order
# -------------------------

# Path to zarrs folder
zarrs_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/zarrs")

# Get all zarr files in order
zarr_files = sorted(zarrs_dir.glob("*.zarr"))
print(f"📁 Found {len(zarr_files)} zarr files")

# Extract nsrrid from each zarr filename (e.g., "shhs1-200002.zarr" -> 200002)
zarr_nsrrids = []
for zarr_file in zarr_files:
    # Extract the number after "shhs1-"
    nsrrid = int(zarr_file.stem.split('-')[-1])
    zarr_nsrrids.append(nsrrid)
    print(f"  {zarr_file.name} -> nsrrid: {nsrrid}")

# Load the original demographics CSV
demographics_df = pd.read_csv('/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel.csv')
print(f"\n📊 Original demographics shape: {demographics_df.shape}")

# Create a mapping dataframe with the desired order
order_df = pd.DataFrame({'nsrrid': zarr_nsrrids, 'order': range(len(zarr_nsrrids))})

# Merge with demographics to get the order column
demographics_ordered = demographics_df.merge(order_df, on='nsrrid', how='inner')

# Sort by the order column
demographics_ordered = demographics_ordered.sort_values('order')

# Drop the order column
demographics_ordered = demographics_ordered.drop('order', axis=1)

print(f"\n✅ Restructured demographics shape: {demographics_ordered.shape}")
print(f"📋 Matched {len(demographics_ordered)} / {len(zarr_nsrrids)} zarr files")

# Show the new order
print(f"\n🔍 First 10 nsrrids in new order:")
print(demographics_ordered['nsrrid'].head(10).tolist())

# Save the restructured CSV
output_path = '/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv'
demographics_ordered.to_csv(output_path, index=False)

print(f"\n💾 Saved restructured CSV to:")
print(f"   {output_path}")

# Verify the order matches
print(f"\n✅ Verification:")
print(f"   Zarr order: {zarr_nsrrids[:5]}")
print(f"   CSV order:  {demographics_ordered['nsrrid'].head(5).tolist()}")
print(f"   Match: {zarr_nsrrids[:5] == demographics_ordered['nsrrid'].head(5).tolist()}")

In [0]:
# ============================================================
# MULTI-GPU UMAP HYPERPARAMETER TUNING (4 GPUs)
# ============================================================
 
import optuna
import cupy as cp
import dask.array as da
import time
import rmm
 
from rmm.allocators.cupy import rmm_cupy_allocator
from cuml.manifold import UMAP as SingleGPUUMAP
from cuml.dask.manifold import UMAP
from cuml.dask.cluster import KMeans
from cuml.metrics.cluster import silhouette_score
 
from dask_cuda import LocalCUDACluster
from dask.distributed import Client
 
# -------------------------------------------------
# Start multi-GPU cluster
# -------------------------------------------------

 
client = Client()

print(client)
client.run(lambda: __import__("dask").config.set({
    "distributed.worker.memory.target": 0.8,
    "distributed.worker.memory.spill": 0.9,
    "distributed.worker.memory.unspill": 0.95,
    "distributed.worker.memory.terminate": 0.98
}))
 

 
# -------------------------------------------------
# Initialize RMM memory pool
# -------------------------------------------------
 
cp.get_default_memory_pool().free_all_blocks()
 
rmm.reinitialize(
    pool_allocator=True,
    initial_pool_size=2 * 1024**3
)
 
cp.cuda.set_allocator(rmm_cupy_allocator)
 
# -------------------------------------------------
# Convert memmap -> distributed GPU array
# -------------------------------------------------
 
print("Creating distributed dataset...")
 
N, D = X_mm.shape
 
X = da.from_array(
    X_mm,
    chunks=(25_000, D)
)
 
X = X.map_blocks(
    lambda x: cp.asarray(x, dtype=cp.float32),
    dtype=cp.float32)

X = client.persist(X)
client.wait(X)

print("Dataset ready (lazy GPU transfer)")

print("Dataset distributed across GPUs")
 
# -------------------------------------------------
# Sample for silhouette evaluation
# -------------------------------------------------
 
rng = np.random.default_rng(42)
eval_size = 200_000
eval_idx = np.sort(rng.choice(N, eval_size, replace=False))
 
# -------------------------------------------------
# Objective function
# -------------------------------------------------
 
def objective(trial):
 
    trial_start = time.time()
 
    n_neighbors = trial.suggest_int("n_neighbors", 10, 80)
    min_dist = trial.suggest_float("min_dist", 0.01, 0.2, log=True)
    n_components = trial.suggest_int("n_components", 5, 20)
    k = trial.suggest_int("k", 3, 16)
 
    # ---------------- UMAP ----------------
 
    t0 = time.time()
    model = SingleGPUUMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric="euclidean",
        random_state=42
    )
 
    #
    reducer = UMAP(
        client=client,
        model=model
    )
 
    embedding = reducer.fit_transform(X)
    embedding = client.persist(embedding)
    client.wait(embedding)
 
    umap_time = time.time() - t0
 
    # ---------------- KMeans ----------------
 
    t0 = time.time()
 
    km = KMeans(
        client=client,
        n_clusters=k,
        max_iter=50,
        random_state=42
    )
 
    labels = km.fit_predict(embedding)
 
    kmeans_time = time.time() - t0
 
    # ---------------- Silhouette ----------------
 
    t0 = time.time()
 
    emb_eval = embedding[eval_idx].persist()
    lbl_eval = labels[eval_idx].persist()

    emb_eval = emb_eval.compute()
    lbl_eval = lbl_eval.compute()
 
    sil = silhouette_score(
        cp.asarray(emb_eval),
        cp.asarray(lbl_eval)
    )
 
    sil_time = time.time() - t0
 
    score = float(sil)
 
    trial_total = time.time() - trial_start
 
    trial.set_user_attr("umap_time_sec", umap_time)
    trial.set_user_attr("kmeans_time_sec", kmeans_time)
    trial.set_user_attr("silhouette_time_sec", sil_time)
    trial.set_user_attr("trial_total_sec", trial_total)
 
    print(
        f"Trial {trial.number:02d} | "
        f"score={score:.4f} | "
        f"UMAP={umap_time:.1f}s | "
        f"KMeans={kmeans_time:.1f}s | "
        f"Sil={sil_time:.1f}s | "
        f"Total={trial_total:.1f}s"
    )
 
    return score
    del embedding
    del labels
    del emb_eval
    del lbl_eval
    del reducer
    del model
    del km
    cp.get_default_memory_pool().free_all_blocks()
# -------------------------------------------------
# Run Optuna
# -------------------------------------------------
 
study = optuna.create_study(direction="maximize")
 
study.optimize(objective, n_trials=20)
 
print("\nBest parameters:", study.best_trial.params)
print("Best silhouette:", study.best_value) 
for t in study.trials:
 
    if t.value is None:
        continue
 
    print(
        f"Trial {t.number:02d} | "
        f"score={t.value:.4f} | "
        f"UMAP={t.user_attrs['umap_time_sec']:.1f}s | "
        f"KMeans={t.user_attrs['kmeans_time_sec']:.1f}s | "
        f"Total={t.user_attrs['trial_total_sec']:.1f}s"
    )

In [0]:
import optuna
import numpy as np
import cupy as cp
import time
import rmm
from dask_cuda import LocalCUDACluster
from dask.distributed import Client
from cuml.manifold import UMAP
from cuml.metrics.cluster import silhouette_score

rmm.reinitialize(
    pool_allocator=True,
    initial_pool_size = 8*1024**3 #8GB pool
)
cp.cuda.set_allocator(rmm.rmm_cupy_allocator)



# -------------------------------------------------

# 1) FIXED SAMPLE

# -------------------------------------------------
 
rng = np.random.default_rng(42)
 
N = X_mm.shape[0]

sample_size = min(1_000_000, N)  # Safe for T4

sample_idx = rng.choice(N, sample_size, replace=False)
 
X_sample = np.take(X_mm, sample_idx, axis=0)

X_sample_gpu = cp.asarray(X_sample, dtype=cp.float32)
 
print(f"Using fixed sample of {sample_size:,} windows")
 
# -------------------------------------------------

# 2) OBJECTIVE FUNCTION

# -------------------------------------------------
 
def objective(trial):
 
    # ---- UMAP params ----

    n_neighbors = trial.suggest_int("n_neighbors", 10, 80)

    min_dist = trial.suggest_float("min_dist", 0.01, 0.2, log=True)

    n_components = trial.suggest_int("n_components", 5, 20)
 
    # ---- KMeans params ----

    k = trial.suggest_int("k", 3, 16)
 
    reducer = UMAP(

        n_neighbors=n_neighbors,

        min_dist=min_dist,

        n_components=n_components,

        metric="euclidean",

        random_state=42,
        build_algo="nn_descent",

        verbose=False,

    )
 
    embedding_gpu = reducer.fit_transform(X_sample_gpu)
 
    km = KMeans(

        n_clusters=k,

        max_iter=100,

        n_init=3,

        random_state=42

    )
 
    labels = km.fit_predict(embedding_gpu)
 
    sil = silhouette_score(embedding_gpu, labels)
 
    score = float(sil)
 
    trial.report(score, step=0)
 
    if trial.should_prune():

        raise optuna.TrialPruned()
 
    return score
 
# -------------------------------------------------

# 3) RUN OPTUNA

# -------------------------------------------------
 
study = optuna.create_study(

    direction="maximize",

    sampler=optuna.samplers.TPESampler(seed=42),

    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)

)
 
study.optimize(objective, n_trials=10)
 
print("\nBest parameters:")

print(study.best_trial.params)

print("Best silhouette:", study.best_value)
 

In [0]:


import optuna

import numpy as np

import cupy as cp

import time
 
from cuml.manifold import UMAP

from cuml.cluster import HDBSCAN

from cuml.metrics.cluster import silhouette_score
 
# -------------------------------------------------

# 1) FIXED SAMPLE (UNCHANGED)

# -------------------------------------------------
 
rng = np.random.default_rng(42)
 
N = X_mm.shape[0]

sample_size = min(1_000_000, N)

sample_idx = rng.choice(N, sample_size, replace=False)
 
X_sample = X_mm[sample_idx]

X_sample_gpu = cp.asarray(X_sample, dtype=cp.float32)
 
print(f"Using fixed sample of {sample_size:,} windows")
 
# -------------------------------------------------

# 2) OBJECTIVE FUNCTION (UMAP + HDBSCAN)

# -------------------------------------------------
 
def objective(trial):
 
    # ---- UMAP params ----

    n_neighbors = trial.suggest_int("n_neighbors", 10, 80)

    min_dist = trial.suggest_float("min_dist", 0.001, 0.2, log=True)

    n_components = trial.suggest_int("n_components", 5, 20)
 
    # ---- HDBSCAN params ----

    min_cluster_size = trial.suggest_int("min_cluster_size", 50, 2000)

    min_samples = trial.suggest_int("min_samples", 10, 500)
 
    reducer = UMAP(

        n_neighbors=n_neighbors,

        min_dist=min_dist,

        n_components=n_components,

        metric="cosine",

        random_state=42,

        verbose=False

    )
 
    embedding_gpu = reducer.fit_transform(X_sample_gpu)
 
    clusterer = HDBSCAN(

        min_cluster_size=min_cluster_size,

        min_samples=min_samples,

        cluster_selection_method="eom",

        metric="euclidean"

    )
 
    labels_gpu = clusterer.fit_predict(embedding_gpu)
 
    # ---- Remove noise for evaluation ----

    mask = labels_gpu != -1
 
    if cp.sum(mask) < 100:

        return -1.0  # Penalize degenerate solutions
 
    unique_clusters = cp.unique(labels_gpu[mask])
 
    if len(unique_clusters) < 2:

        return -1.0  # Need at least 2 clusters
 
    sil = silhouette_score(

        embedding_gpu[mask],

        labels_gpu[mask]

    )
 
    score = float(sil)
 
    trial.report(score, step=0)
 
    if trial.should_prune():

        raise optuna.TrialPruned()
 
    return score
 
# -------------------------------------------------

# 3) RUN OPTUNA

# -------------------------------------------------
 
study = optuna.create_study(

    direction="maximize",

    sampler=optuna.samplers.TPESampler(seed=42),

    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)

)
 
study.optimize(objective, n_trials=30)
 
print("\nBest parameters:")

print(study.best_trial.params)

print("Best silhouette:", study.best_value)

 

In [0]:
import cupy as cp

import numpy as np

from cuml.manifold import UMAP
 
# ----------------------------

# Subsample windows

# ----------------------------
 
rng = np.random.default_rng(42)

sample_size = min(1_000_000, len(X_mm))

sample_idx = rng.choice(len(X_mm), sample_size, replace=False)
 
X_sample = X_mm[sample_idx]

X_sample_gpu = cp.asarray(X_sample, dtype=cp.float32)
 
print(f"UMAP sample size: {sample_size:,}")
 
# ----------------------------

# Run UMAP (10D)

# ----------------------------
 
umap_model = UMAP(

    n_neighbors=79,

    min_dist=0.009870615836067072,

    n_components=12,

    metric="euclidean",

    random_state=42,

    verbose=True,

)
 
embedding_gpu = umap_model.fit_transform(X_sample_gpu)

embedding = cp.asnumpy(embedding_gpu)
 
print("UMAP complete:", embedding.shape)
 

In [0]:
from cuml.cluster import KMeans

from cuml.metrics.cluster import silhouette_score
 
embedding_gpu = cp.asarray(embedding, dtype=cp.float32)
 
best_k = 13


print(f"\nTesting k={best_k} in UMAP space...")
 
km = KMeans(

    n_clusters=best_k,

    max_iter=100,

    n_init=5,

    random_state=42,
    
    verbose=True,

    )
 
labels = km.fit_predict(embedding_gpu)
 
sil = silhouette_score(embedding_gpu, labels)

print(f"Silhouette for k = {best_k}: {float(sil):.4f}")
 


In [0]:
from cuml.metrics.cluster import silhouette_score
mask = labels_hdb != -1
if mask.sum() > 0:
    sil_hdb = silhouette_score(
        embedding_gpu[mask],
        labels_hdb[mask]
    )
    print("HDBSCAN silhouette (excluding noise):", float(sil_hdb))
 

In [0]:
# ------------------------------------------------------------

# 2) RANDOM HYPERPLANE LSH (TIGHTENED + SAMPLE-BASED)

# ------------------------------------------------------------
 
# Use memmap-backed matrix

X = X_mm

N, D = X.shape
 
# -----------------------------------

# Use subsample for tuning

# -----------------------------------
 
rng = np.random.default_rng(42)

sample_size = min(5_000_000, N)

sample_idx = rng.choice(N, sample_size, replace=False)

X_sample = X[sample_idx]
 
print(f"Tuning LSH on sample of {sample_size:,} points")
 
# -----------------------------------

# LSH function

# -----------------------------------
 
def lsh_hash(X, n_bits, seed=42):

    rng = np.random.default_rng(seed)

    hyperplanes = rng.standard_normal((n_bits, D)).astype(np.float32)

    proj = X @ hyperplanes.T

    bits = (proj > 0)

    return np.packbits(bits, axis=1)
 
# -----------------------------------

# Bucket statistics

# -----------------------------------
 
def bucket_stats(hash_vals):

    # Convert bit arrays to compact representation

    unique, counts = np.unique(hash_vals, axis=0, return_counts=True)
 
    total = counts.sum()
 
    return {

        "nonempty": len(unique),

        "singleton_frac": float(np.sum(counts == 1)) / total,

        "median_size": float(np.median(counts)),

        "max_size": int(np.max(counts)),

    }
 
# -----------------------------------

# Bit depth search

# -----------------------------------
 
bits_range = range(6, 25)   # narrower, realistic range

target_buckets = 2000
 
best_bits = None

best_score = np.inf
 
for b in bits_range:
 
    h = lsh_hash(X_sample, n_bits=b)

    stats = bucket_stats(h)
 
    # Balanced scoring

    score = (

        abs(stats["nonempty"] - target_buckets)

        + 2000 * stats["singleton_frac"]

        + 0.001 * stats["max_size"]  # mild penalty for collapse

    )
 
    print(

        f"bits={b:2d} | "

        f"buckets={stats['nonempty']:4d} | "

        f"singleton_frac={stats['singleton_frac']:.3f} | "

        f"median={stats['median_size']:.1f} | "

        f"max={stats['max_size']}"

    )
 
    if score < best_score:

        best_score = score

        best_bits = b
 
print("\nChosen LSH bits:", best_bits)
 

In [0]:
# ------------------------------------------------------------
# 4) LSH-MEANS INITIALIZATION (TIGHTENED)
# ------------------------------------------------------------

 
def lsh_means_init(X, n_clusters, n_bits, n_rounds=3, min_bucket_size=50):

    """

    Improved LSH-means initialization.

    - Multiple hashing rounds for stability

    - Remove tiny buckets

    - Pre-cluster bucket centroids

    """

    import numpy as np

    from sklearn.cluster import KMeans as SKKMeans

    N, D = X.shape

    all_centroids = []
 
    for r in range(n_rounds):

        h = lsh_hash(X, n_bits=n_bits, seed=42 + r)

        unique, inv, counts = np.unique(h, return_inverse=True, return_counts=True)
 
        for idx, count in zip(unique, counts):

            if count < min_bucket_size:

                continue

            mask = (h == idx)

            centroid = X[mask].mean(axis=0)

            all_centroids.append(centroid)
 
    if len(all_centroids) < n_clusters:

        raise ValueError("Not enough stable buckets found. Increase n_bits or reduce min_bucket_size.")
 
    C = np.vstack(all_centroids)
 
    # Pre-cluster centroids down to desired k

    sk = SKKMeans(n_clusters=n_clusters, n_init=10, random_state=42)

    sk.fit(C)
 
    return sk.cluster_centers_
 

In [0]:
# ------------------------------------------------------------
# FAST CORRECT K SWEEP
# ------------------------------------------------------------
 
from cuml.cluster import KMeans
from cuml.metrics.cluster import silhouette_score
import cupy as cp
import numpy as np
import time
 
X = X_mm
N, D = X.shape
 
# ----------------------------
# Sample once
# ----------------------------
 
rng = np.random.default_rng(42)
sample_size = min(500_000, N)
sample_idx = rng.choice(N, sample_size, replace=False)
 
X_sample = X[sample_idx]
X_sample_gpu = cp.asarray(X_sample, dtype=cp.float32)
 
print(f"Using {sample_size:,} points for K selection")
 
# ----------------------------
# Compute LSH ON SAMPLE ONLY
# ----------------------------
 
print("Computing LSH on sample only...")
h_sample = lsh_hash(X_sample, n_bits=best_bits)
 
unique_hashes, inverse, counts = np.unique(
    h_sample, axis=0, return_inverse=True, return_counts=True
)
 
bucket_order = np.argsort(counts)[::-1]
 
k_values = list(range(19, 25))
results = []
 
for k in k_values:
 
    start = time.time()
    print(f"\nTesting k={k}...")
 
    # Build centroids WITHOUT rehashing
    centroids = []
    for idx in bucket_order:
        if counts[idx] < 20:
            continue
        mask = (inverse == idx)
        centroids.append(X_sample[mask].mean(axis=0))
        if len(centroids) >= k:
            break
 
    C_gpu = cp.asarray(np.vstack(centroids[:k]), dtype=cp.float32)
 
    km = KMeans(
        n_clusters=k,
        init=C_gpu,
        max_iter=50,
        verbose=True,
        n_init=1,
        random_state=42,
    )
 
    labels = km.fit_predict(X_sample_gpu)
 
    # ---- GPU silhouette (NO .get()) ----
    sil = silhouette_score(X_sample_gpu, labels)
 
    elapsed = time.time() - start
    print(f"  silhouette={float(sil):.4f}  time={elapsed:.1f}s")
 
    results.append((k, float(sil)))
 
best_k, best_sil = max(results, key=lambda x: x[1])
 
print(f"\nChosen k={best_k} (silhouette={best_sil:.4f})")


In [0]:
best_k = 12

In [0]:
from cuml.cluster import KMeans

X = X_mm
N, D = X.shape
 
print("Moving full dataset to GPU...")
X_gpu = cp.asarray(X, dtype=cp.float32)
 
print("Computing LSH init on full dataset...")
 
# Hash full dataset once
h_full = lsh_hash(X, n_bits=best_bits)
 
unique_hashes, inverse, counts = np.unique(
    h_full, axis=0, return_inverse=True, return_counts=True
)
 
bucket_order = np.argsort(counts)[::-1]
 
centroids = []
for idx in bucket_order:
    if counts[idx] < 50:
        continue
    mask = (inverse == idx)
    centroids.append(X[mask].mean(axis=0))
    if len(centroids) >= best_k:
        break
 
C_gpu = cp.asarray(np.vstack(centroids[:best_k]), dtype=cp.float32)
 
print("Fitting final KMeans...")
t0 = time.time()
 
final_km = KMeans(
    n_clusters=best_k,
    init=C_gpu,
    max_iter=200,
    n_init=1,
    random_state=42,
)
 
cluster_id_gpu = final_km.fit_predict(X_gpu)
cluster_id = cp.asnumpy(cluster_id_gpu)
 
print(f"Final fit complete in {time.time() - t0:.1f}s")

In [0]:
np.save(cache_dir / f"cluster_id__{tag}.npy", cluster_id)
print("Cluster IDs saved.")
 
unique, counts = np.unique(cluster_id, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Cluster {u}: {c:,} windows")
 

In [0]:
df = pd.DataFrame({
    "cluster": cluster_id,
    "nsrrid": zarr_file_idx
})
 
cluster_fraction = (
    df.groupby("nsrrid")["cluster"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)

cluster_fraction.index = zarr_nsrrids
cluster_fraction.index.name = 'nsrrid'

print(cluster_fraction.index[:10])

In [0]:
demographics_df = pd.read_csv("/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv")
demographics_df["nsrrid"] = demographics_df["nsrrid"].astype(int)

In [0]:
cluster_demo = (
    cluster_fraction
    .reset_index()
    .merge(demographics_df, on="nsrrid", how="left")
) 
print("Merged subjects:", len(cluster_demo))
print("Unique nsrrid:", cluster_demo["nsrrid"].nunique())
 

In [0]:
cluster_demo["dominant_cluster"] = (cluster_fraction.idxmax(axis=1).values)

In [0]:
def summarize_cluster(df):
    return {
        "N": len(df),
        "Age_mean": df["age_s1"].mean(),
        "Age_min": df["age_s1"].min(),
        "Age_max": df["age_s1"].max(),
        "BMI_mean": df["bmi_s1"].mean(),
        "BMI_min": df["bmi_s1"].min(),
        "BMI_max": df["bmi_s1"].max(),
        "AHI_mean": df["ahi_a0h4_s1"].mean(),
        "AHI_min": df["ahi_a0h4_s1"].min(),
        "AHI_max": df["ahi_a0h4_s1"].max(),
        "ESS_mean": df["ess_s1"].mean(),
        "MinSat_mean": df["MinSat"].mean(),
        "PctLT90_mean": df["pctlt90"].mean(),
        "Diabetes_%": (df["Diabetes"] == "Yes").mean() * 100,
        "Prev_CVD_%": (df["prev_cvd_all_01"] == "Yes").mean() * 100,
        "Male_%": (df["gender"] == "Male").mean() * 100,
        "Smoker_%": (df["smokecat_s1"] == "Current").mean() * 100,
    }
 
summary_table = (
    cluster_demo
    .groupby("dominant_cluster")
    .apply(summarize_cluster)
    .apply(pd.Series)
)
 
summary_table
output_path = Path("/Volumes/kumc_sleep/sleep_studies/shhs_data/clustering_outputs/")
summary_table.to_csv(output_path / f"cluster_summary__{tag}.csv")
print(f"Saved to: {output_path}")
# ------------------------------------------------------------

In [0]:
demographics_path = "/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv"
 
demographics_df = pd.read_csv(demographics_path)
 
print("Loaded demographics:", demographics_df.shape)
print(demographics_df.columns)

In [0]:
# ============================================================

# UMAP Visualization of Clustered 6-Second Physiological States

# (2D scatter + density hexbin in one run)

# ============================================================
 
import numpy as np

import cupy as cp

import matplotlib.pyplot as plt

from cuml.manifold import UMAP
 
# ----------------------------

# 1) Subsample windows

# ----------------------------
 
rng = np.random.default_rng(42)

viz_size = min(200_000, len(cluster_id))  # safe range: 150k–250k
 
viz_idx = rng.choice(len(cluster_id), viz_size, replace=False)
 
X_viz = X_mm[viz_idx]

labels_viz = cluster_id[viz_idx]
 
print(f"Visualization sample size: {viz_size:,}")
 
# ----------------------------

# 2) Move to GPU

# ----------------------------
 
X_viz_gpu = cp.asarray(X_viz, dtype=cp.float32)
 
# ----------------------------

# 3) Run GPU UMAP

# ----------------------------
 
reducer = UMAP(

    n_neighbors=15,

    min_dist=0.1,

    n_components=2,

    metric="euclidean",

    random_state=42,

)
 
embedding_gpu = reducer.fit_transform(X_viz_gpu)

embedding = cp.asnumpy(embedding_gpu)

# ------------------------------------------------------------

# Correctly map window -> subject -> AHI

# ------------------------------------------------------------
 
# Step 1: get file indices for sampled windows

window_file_idx = zarr_file_idx[viz_idx]
 
# Step 2: convert file_idx -> nsrrid

window_nsrrids = np.array(zarr_nsrrids)[window_file_idx]
 
# Step 3: build fast lookup table

ahi_lookup = cluster_demo.set_index("nsrrid")["ahi_a0h4_s1"].to_dict()
 
# Step 4: map nsrrid -> AHI

ahi_viz = np.array([

    ahi_lookup.get(nsrrid, np.nan)

    for nsrrid in window_nsrrids

])
 
print("AHI mapping complete.")
 

print("UMAP complete.")
 
# ----------------------------

# 4) 2D Scatter Plot

# ----------------------------
 
plt.figure(figsize=(10, 8))

scatter = plt.scatter(

    embedding[:, 0],

    embedding[:, 1],

    c=labels_viz,

    s=3,

    cmap="tab10",

    alpha=0.7,

)
 
plt.title(f"UMAP of 6-Second Physiological States (k={best_k})")

plt.xlabel("UMAP-1")

plt.ylabel("UMAP-2")

plt.colorbar(scatter, label="Cluster")

plt.tight_layout()

plt.show()
 
# ----------------------------

# 5) Density Hexbin Plot

# ----------------------------
 
plt.figure(figsize=(10, 8))

hb = plt.hexbin(

    embedding[:, 0],

    embedding[:, 1],

    gridsize=150,

    C=labels_viz,

    reduce_C_function=np.mean,

    cmap="viridis",

)
 
plt.colorbar(hb, label="Mean Cluster ID")

plt.title("UMAP Density View")

plt.xlabel("UMAP-1")

plt.ylabel("UMAP-2")

plt.tight_layout()

plt.show()


# ------------------------------------------------------------

# Color UMAP by AHI (clinical gradient check)

# ------------------------------------------------------------
 
# Map window -> nsrrid

window_nsrrids = np.array(zarr_nsrrids)[zarr_file_idx[viz_idx]]
 
# Build lookup from nsrrid to AHI

ahi_lookup = cluster_demo.set_index("nsrrid")["ahi_a0h4_s1"]
 
ahi_viz = np.array([

    ahi_lookup.get(nsrrid, np.nan)

    for nsrrid in window_nsrrids

])
 
plt.figure(figsize=(10, 8))
plt.scatter(
    embedding[:, 0],
    embedding[:, 1],
    c=ahi_viz,
    s=3,
    cmap="viridis",
    alpha=0.7
)
plt.colorbar(label="AHI")
plt.title("UMAP Colored by AHI")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.show()

In [0]:
cluster_demo.groupby("dominant_cluster").size()

# GPU-Accelerated Clustering Complete ✅

## Workflow Summary

This notebook performs GPU-accelerated clustering on latent representations extracted by the **Representation_Extracting** notebook.

### Pipeline Steps

1. **Load Cached Latents** (Cell 5)
   - Loads `Z`, `night_id`, `time_idx`, `zarr_file_idx`, `windows_start_idx`, `zarr_files_list`
   - Loads `zarr_id_map.json` for file mapping

2. **Normalize** (Cell 7)
   - L2 normalization for angular geometry

3. **LSH Hashing** (Cell 8)
   - Random hyperplane LSH to find optimal bit depth
   - Targets ~2000 buckets for initialization

4. **LSH-Means Initialization** (Cell 9)
   - Uses LSH buckets to create smart initial centroids
   - **GPU-accelerated KMeans** for centroid refinement

5. **K-Value Sweep** (Cell 10)
   - Tests k values from 8-17
   - **GPU-accelerated KMeans** for each k
   - Evaluates using silhouette score

6. **Final Clustering** (Cell 11)
   - **GPU-accelerated KMeans** with best k
   - Assigns cluster IDs to all windows

7. **UMAP Visualization** (Cell 12)
   - 3D UMAP embedding colored by cluster

8. **Save Results** (Cell 13)
   - Creates comprehensive CSV with cluster assignments
   - Maps back to source zarr files
   - Includes all metadata for further analysis

## Output Files

**Saved to**: `cache_dir / cluster_assignments__{tag}.csv`

**Columns**:
- `window_idx`: Sequential window index
- `cluster_id`: Assigned cluster ID (from GPU KMeans)
- `zarr_file_idx`: Numeric zarr file ID
- `night_id`: Night/batch identifier
- `time_idx_sec`: Time index in seconds
- `windows_start_idx`: Start index in original zarr
- `zarr_uid`: Original zarr file UID
- `zarr_file_path`: Full path to source zarr file

## GPU Acceleration

✅ **All clustering operations use cuML GPU KMeans**
- Faster than CPU sklearn by 10-100x on large datasets
- Runs on T4 GPU (Standard_NC4as_T4_v3)
- Compatible with LSH-means initialization strategy

## Next Steps

Use the `cluster_assignments__{tag}.csv` file to:
- Analyze cluster characteristics
- Extract representative windows from each cluster
- Map clusters back to original zarr files for detailed inspection
- Perform downstream analysis on specific clusters